# Genetic Algorithm Meal Planner

In [1]:
from models import Pantry, Recipe
from mock_pantry import create_mock_pantry, create_mock_recipes, create_default_preferences, INGREDIENT_UNIT_COSTS
from datetime import datetime
from GAMealPlan import evaluate_meal_plan
from pygad import GA

In [2]:
pantry = create_mock_pantry(num_ingredients = 15, random_state = 1)
recipes = create_mock_recipes()
preferences = create_default_preferences()

current_date = datetime.now()

In [3]:
pantry_stock = pantry.stock
days_until_expiry = {ingredient.name: max(0, (ingredient.estimated_expiration_date - current_date).days) for ingredient in pantry.ingredients}

In [4]:
def fitness_function(_ga_instance, solution, _solution_index):
    return evaluate_meal_plan(
        solution,
        recipes,
        pantry_stock,
        days_until_expiry,
        preferences,
        INGREDIENT_UNIT_COSTS,
    )

In [5]:
GENERATION_PRINT_INTERVAL = 25

In [6]:
def on_generation(ga_instance):
    num_generations = ga_instance.generations_completed

    if num_generations % GENERATION_PRINT_INTERVAL == 0:
        best_fitness = ga_instance.best_solution() [1]
        print(f"Generation {num_generations}, Best Fitness: {best_fitness:.2f}")

In [7]:
NUM_GENERATIONS = 200
NUM_PARENTS_MATING = 20
POPULATION_SIZE = 100
NUM_GENES = 21 # 3 meals/day x 7 days

In [8]:
ga_instance = GA(
    num_generations = NUM_GENERATIONS,
    num_parents_mating = NUM_PARENTS_MATING,
    fitness_func = fitness_function,
    sol_per_pop = POPULATION_SIZE,
    num_genes = NUM_GENES,
    gene_type = int,
    gene_space = list(range(len(recipes))),
    parent_selection_type = "tournament",
    K_tournament = 5,
    keep_elitism = 1,
    crossover_type = "single_point",
    crossover_probability = 0.8,
    mutation_type = "random",
    mutation_probability = 0.1,
    on_generation = on_generation,
    random_seed = 1
)

In [9]:
ga_instance.run()

Generation 25, Best Fitness: -671.92
Generation 50, Best Fitness: -633.55
Generation 75, Best Fitness: -623.79
Generation 100, Best Fitness: -618.76
Generation 125, Best Fitness: -618.76
Generation 150, Best Fitness: -618.76
Generation 175, Best Fitness: -616.96
Generation 200, Best Fitness: -616.96


In [10]:
best_meal_plan, best_fitness, _ = ga_instance.best_solution()

print(f"Best fitness score: {best_fitness:.2f}")

Best fitness score: -616.96


In [11]:
for day in range(7):
    print(f"\nDay {day + 1} meal plan:")

    for meal in range(3):
        recipe = recipes [int(best_meal_plan [day * 3 + meal])]

        print(f"Meal {meal + 1}: {recipe.name}")


Day 1 meal plan:
Meal 1: Bell Pepper Frittata
Meal 2: Bell Pepper Frittata
Meal 3: Greek Salad

Day 2 meal plan:
Meal 1: Bell Pepper Frittata
Meal 2: Bell Pepper Frittata
Meal 3: Bell Pepper Frittata

Day 3 meal plan:
Meal 1: Bell Pepper Frittata
Meal 2: Bell Pepper Frittata
Meal 3: Bell Pepper Frittata

Day 4 meal plan:
Meal 1: Bell Pepper Frittata
Meal 2: Bell Pepper Frittata
Meal 3: Bell Pepper Frittata

Day 5 meal plan:
Meal 1: Bell Pepper Frittata
Meal 2: Scrambled Eggs
Meal 3: Bell Pepper Frittata

Day 6 meal plan:
Meal 1: Bell Pepper Frittata
Meal 2: Bell Pepper Frittata
Meal 3: Greek Salad

Day 7 meal plan:
Meal 1: Bell Pepper Frittata
Meal 2: Bell Pepper Frittata
Meal 3: Greek Salad


In [12]:
# going through the best meal plan and calculating how much of each ingredient is consumed from the pantry

consumed_stock = dict.fromkeys(pantry_stock.keys(), 0)

for index in best_meal_plan:
    recipe = recipes [int(index)]

    for ingredient_name, quantity_needed in recipe.ingredient_quantities.items():
        available = pantry_stock.get(ingredient_name, 0) - consumed_stock.get(ingredient_name, 0)
        from_pantry = max(0, min(available, quantity_needed))
        consumed_stock [ingredient_name] = consumed_stock.get(ingredient_name, 0) + from_pantry


In [26]:
print("Pantry usage summary:")
print(f"  {chr(39)}Ingredient{chr(39):<16}  Available  Consumed  Unused  Expires in")
print("  " + "-" * 68)

for ingredient in pantry.ingredients:
    available = pantry_stock [ingredient.name]
    used = consumed_stock.get(ingredient.name, 0)
    unused = max(0, available - used)
    days = days_until_expiry [ingredient.name]
    expiry_str = str(days) + "d"
    flag = " !" if days <= 3 and unused > 0 else ""
    print(f"  {ingredient.name:<20} {available:>10} {used:>10} {unused:>9} {expiry_str:>9}{flag}")


Pantry usage summary:
  'Ingredient'                 Available  Consumed  Unused  Expires in
  --------------------------------------------------------------------
  chicken_breast                1          0         1       17d
  bell_pepper                   2          2         0       24d
  lemon                         3          3         0        1d
  milk                          1          1         0        2d
  garlic                        3          0         3       23d
  butter                        1          1         0       11d
  pasta                         1          0         1       14d
  tomato                        2          2         0       27d
  onion                         1          1         0       18d


In [14]:
print("Nutritional information summary:\n")
print(f"\t{'Day':<5} {'Meal 1':<20} {'Meal 2':<20} {'Meal 3':<20} {'Calories':>10} {'Protein':>10}")
print("\t" + "-" * 95)

for day in range(7):
    meal_indices = best_meal_plan [day * 3 : day * 3 + 3]
    meal_names = [recipes [int(index)].name for index in meal_indices]
    total_calories = sum(recipes [int(index)].nutritional_info.calories for index in meal_indices)
    total_protein = sum(recipes [int(index)].nutritional_info.protein for index in meal_indices)

    print(f"\t{day + 1:<5} {meal_names[0]:<20} {meal_names[1]:<20} {meal_names[2]:<20} {total_calories:>10.2f} {total_protein:>10.2f}")

Nutritional information summary:

	Day   Meal 1               Meal 2               Meal 3                 Calories    Protein
	-----------------------------------------------------------------------------------------------
	1     Bell Pepper Frittata Bell Pepper Frittata Greek Salad             1398.36      87.50
	2     Bell Pepper Frittata Bell Pepper Frittata Bell Pepper Frittata    1769.40     124.65
	3     Bell Pepper Frittata Bell Pepper Frittata Bell Pepper Frittata    1769.40     124.65
	4     Bell Pepper Frittata Bell Pepper Frittata Bell Pepper Frittata    1769.40     124.65
	5     Bell Pepper Frittata Scrambled Eggs       Bell Pepper Frittata    1659.78     114.79
	6     Bell Pepper Frittata Bell Pepper Frittata Greek Salad             1398.36      87.50
	7     Bell Pepper Frittata Bell Pepper Frittata Greek Salad             1398.36      87.50
